# Oracles, Manipulation, and Flash Loans — Perfectly Correct, Completely Robbed

A blockchain is a closed room. It can be extremely sure about the papers on its own table. It cannot see a football score, a weather station, or the price of ETH on an exchange. An **oracle** is the courier that brings an outside fact inside, in a form a contract can use.

**Question:** how can a protocol execute every instruction correctly and still lose money? We will watch a contract trust a reported price, then an AMM's momentary reserves, then give an attacker a flash loan and one atomic transaction. The arithmetic is real; the actors are imaginary, so nobody needs to call a lawyer.

**Scope:** this is a deterministic teaching model, not a recipe for attacking or operating a real protocol. It omits fees, liquidity providers, slippage limits, token transfers, external arbitrage, and real-world legal or economic consequences.


## Recap

| Notebook | What it established |
| --- | --- |
| 1–4 | Hash links, stake, Merkle proofs, and permissioned membership. |
| 5 | `Transaction` and `Network`: each node has its own mempool; gossip is imperfect; a proposer can only include what it has heard. |

This notebook **keeps those classes**. An oracle report and a flash-loan heist are still transactions. They still have to be gossiped. They still have to be included. The new question is why a contract should treat the *payload* as the price of ETH.

A contract cannot discover market truth by itself. It must trust some input — a named feed, a median of feeds, or an on-chain AMM spot price that an attacker can temporarily shove around.


## 1. Oracles, quickly

A smart contract can inspect its own state and the inputs you hand it. It cannot check an exchange screen. Someone has to *tell* it. Consensus notarises what it was given. It does not fact-check the universe.

Meet the loan we will keep bullying: **10 ETH collateral, $12,000 debt**. At a sensible ~$2,000/ETH the collateral ratio is 1.67, above a 1.5 liquidation threshold. Healthy — until the price source lies, or until a puddle is mistaken for the ocean.

The next cell is the report type, a median aggregator, and the ratio helpers. Then we feed the loan a liar, then a median.

> Pause and predict: if the contract trusts one reported price of $1,200, what happens to this loan? What if that lie sits next to two reports near $2,000?


In [4]:
import statistics
from dataclasses import dataclass


@dataclass(frozen=True)
class PriceReport:
    """One reported ETH/USD price from a named source.

    Attributes:
        source: Who reported this price. Nothing here authenticates them.
        eth_price_usd: The reported price.
    """

    source: str
    eth_price_usd: float


def collateral_ratio(
    collateral_eth: float, debt_usd: float, eth_price_usd: float
) -> float:
    """Return collateral value divided by debt, at a given ETH price."""
    if collateral_eth <= 0 or debt_usd <= 0 or eth_price_usd <= 0:
        raise ValueError("Collateral, debt, and price must be positive.")
    return collateral_eth * eth_price_usd / debt_usd


def should_liquidate(ratio: float, threshold: float = 1.5) -> bool:
    """Return whether a collateral ratio is below the liquidation threshold."""
    return ratio < threshold


class MedianOracle:
    """A price source that reports the median of several independent reports."""

    def __init__(self, reports: list[PriceReport]) -> None:
        if not reports:
            raise ValueError("At least one price report is required.")
        self.reports = reports

    @property
    def price(self) -> float:
        """Median reported price — resists a single outlier report."""
        return statistics.median(report.eth_price_usd for report in self.reports)


In [5]:
collateral_eth = 10
debt_usd = 12_000
bad_report = PriceReport("malicious-feed", 1_200)
bad_report_ratio = collateral_ratio(
    collateral_eth, debt_usd, bad_report.eth_price_usd
)

print(f"Position: {collateral_eth} ETH collateral / ${debt_usd:,} debt")
print(f"Single report: ${bad_report.eth_price_usd:,.0f}/ETH")
print(f"Single-source collateral ratio: {bad_report_ratio:.2f}")
if should_liquidate(bad_report_ratio):
    print("Single-source verdict: LIQUIDATE (wrong)")

reports = [
    PriceReport("independent-feed-a", 2_005),
    PriceReport("malicious-feed", 1_200),
    PriceReport("independent-feed-b", 1_995),
]
median_oracle = MedianOracle(reports)
median_price = median_oracle.price
median_report_ratio = collateral_ratio(collateral_eth, debt_usd, median_price)

print()
print("Reports: $2,005, $1,200, $1,995 per ETH")
print(f"Median report: ${median_price:,.0f}/ETH")
print(f"Median collateral ratio: {median_report_ratio:.2f}")
if not should_liquidate(median_report_ratio):
    print("Median verdict: HEALTHY")


Position: 10 ETH collateral / $12,000 debt
Single report: $1,200/ETH
Single-source collateral ratio: 1.00
Single-source verdict: LIQUIDATE (wrong)

Reports: $2,005, $1,200, $1,995 per ETH
Median report: $1,995/ETH
Median collateral ratio: 1.66
Median verdict: HEALTHY


**Read the result:** the false $1,200 price values 10 ETH at $12,000, so the ratio is 1.00 and the rule liquidates a healthy position. Correct arithmetic, wrong universe. The median ignores the outlier, keeps $1,995, and the ratio 1.66 stays healthy.

Aggregation raises the cost of lying. It does not prove reality, and it does not help if the protocol later ignores the median and reads a thin pool instead.

That is the next trick. An AMM spot price already lives on-chain. The reserves are real. The formula `USD / ETH` is real. And it is still an oracle: the contract is treating one observation as "what ETH is worth." A puddle is not the ocean.


## 2. The whole attack, before the code

No lying courier required. The pool will tell the truth about *itself*, and that truth will be a terrible proxy for "the market."

Inside **one atomic transaction**, the attacker will: (1) borrow ETH, (2) dump it into a small ETH/USD pool, (3) liquidate a victim whom the distorted price makes look unsafe, (4) buy the borrowed ETH back, (5) repay the principal, and (6) keep the seized collateral. Flash loans do not create the bug. They just mean the attacker does not need to be rich *before* lunch.

Atomic means all-or-nothing: if any step raises, the pool, loans, and lender balance rewind. A perfectly atomic robbery is still a robbery. On Ethereum a reverted transaction can still be included and consume gas; **included is not the same as succeeded**.

That "one atomic transaction" is also the answer to notebook 5's public waiting room. If dump and liquidate were two gossiped `Transaction`s, other nodes would see the cheap puddle between them. Bundling the steps means the mempool shows one payload, and inclusion executes all of it — or none.

| Stage | What changes | Why it matters |
| --- | --- | --- |
| Borrow | attacker temporarily receives ETH | capital is available only for this transaction |
| Dump | AMM spot price falls | the lending rule sees a misleading input |
| Liquidate | attacker pays debt and receives collateral | the vulnerable rule follows its price source |
| Buy back and repay | borrowed principal returns to lender | any remaining ETH belongs to the attacker |


## 3. The AMM: a puddle with a price

A constant-product AMM keeps `x * y = k`. Here `x` is ETH reserve and `y` is USD reserve, so the displayed spot price is `USD reserve / ETH reserve`. That number is not a journalist. It is the current ratio of two piles of tokens. We omit fees, slippage limits, and external arbitrage so the splash is visible.

We start with 50 ETH and $100,000 ($2,000/ETH) and sell 1 ETH and 40 ETH into **separate fresh pools**. A pebble and a boulder, same puddle, different splash.

> Pause and predict: which sale moves the displayed price more, and does `x * y` remain essentially unchanged?


In [9]:
from dataclasses import dataclass


@dataclass
class AMMPool:
    """A constant-product (x*y=k) two-asset market maker.

    Attributes:
        eth_reserve: ETH held by the pool.
        usd_reserve: USD held by the pool.
    """

    eth_reserve: float
    usd_reserve: float

    def __post_init__(self) -> None:
        if self.eth_reserve <= 0 or self.usd_reserve <= 0:
            raise ValueError("AMM reserves must be positive.")

    @property
    def spot_price(self) -> float:
        """Current displayed price: USD reserve per unit of ETH reserve."""
        return self.usd_reserve / self.eth_reserve

    @property
    def constant_product(self) -> float:
        """The invariant `x * y` a swap should preserve (there are no fees here)."""
        return self.eth_reserve * self.usd_reserve

    def swap_eth_for_usd(self, eth_in: float) -> float:
        """Sell ETH into the pool, moving both reserves and the spot price.

        Args:
            eth_in: ETH sold into the pool. Must be positive.

        Returns:
            USD received in exchange.

        Raises:
            ValueError: If ``eth_in`` is not positive.
        """
        if eth_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_eth_reserve = self.eth_reserve + eth_in
        new_usd_reserve = k / new_eth_reserve
        usd_out = self.usd_reserve - new_usd_reserve
        self.eth_reserve, self.usd_reserve = new_eth_reserve, new_usd_reserve
        return usd_out

    def swap_usd_for_eth(self, usd_in: float) -> float:
        """Sell USD into the pool, moving both reserves and the spot price.

        Args:
            usd_in: USD sold into the pool. Must be positive.

        Returns:
            ETH received in exchange.

        Raises:
            ValueError: If ``usd_in`` is not positive.
        """
        if usd_in <= 0:
            raise ValueError("Swap input must be positive.")
        k = self.constant_product
        new_usd_reserve = self.usd_reserve + usd_in
        new_eth_reserve = k / new_usd_reserve
        eth_out = self.eth_reserve - new_eth_reserve
        self.usd_reserve, self.eth_reserve = new_usd_reserve, new_eth_reserve
        return eth_out

In [10]:
initial_pool = AMMPool(50.0, 100_000.0)
small_trade_pool = AMMPool(50.0, 100_000.0)
large_trade_pool = AMMPool(50.0, 100_000.0)

small_usd_out = small_trade_pool.swap_eth_for_usd(1.0)
large_usd_out = large_trade_pool.swap_eth_for_usd(40.0)
small_impact = (small_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
large_impact = (large_trade_pool.spot_price / initial_pool.spot_price - 1) * 100
constant_product_preserved = abs(
    large_trade_pool.constant_product - initial_pool.constant_product
) < 1e-6

print(
    f"Initial pool: {initial_pool.eth_reserve:.2f} ETH and "
    f"${initial_pool.usd_reserve:,.2f}; spot price ${initial_pool.spot_price:,.2f}/ETH"
)
print(
    f"Small 1 ETH sale: receives ${small_usd_out:,.2f}; "
    f"price ${small_trade_pool.spot_price:,.2f}/ETH ({small_impact:.2f}%)"
)
print(
    f"Large 40 ETH sale: receives ${large_usd_out:,.2f}; "
    f"price ${large_trade_pool.spot_price:,.2f}/ETH ({large_impact:.2f}%)"
)
print(f"x * y preserved within floating-point tolerance: {constant_product_preserved}")


Initial pool: 50.00 ETH and $100,000.00; spot price $2,000.00/ETH
Small 1 ETH sale: receives $1,960.78; price $1,922.34/ETH (-3.88%)
Large 40 ETH sale: receives $44,444.44; price $617.28/ETH (-69.14%)
x * y preserved within floating-point tolerance: True


**Read the result:** the 40 ETH sale moves the displayed price by about 69%, far more than the 1 ETH sale. Nothing says ETH became 69% cheaper everywhere; this thin pool observed its own reserves. The AMM is an honest reporter of a local puddle. The constant product stays stable apart from ordinary floating-point rounding.


## 4. The vulnerable lending protocol

The protocol liquidates whenever collateral value / debt falls below 1.5. Its mistake is intentionally narrow: it reads `pool.spot_price` directly. That is an oracle choice, even though nobody named it "the oracle."

Real protocols also wrestle with bonuses, partial liquidations, fees, and token transfers. We omit those so the bad price source is the only moving part. Same 10 ETH / $12,000 loan as before.

> Pause and predict: at $2,000/ETH, is this position above or below a 1.5 threshold?


In [13]:
@dataclass(frozen=True)
class Loan:
    """A borrower's position: collateral posted against debt owed.

    Attributes:
        collateral_eth: ETH locked as collateral. Must be positive.
        debt_usd: Outstanding debt in USD. Must be positive.
    """

    collateral_eth: float
    debt_usd: float

    def __post_init__(self) -> None:
        if self.collateral_eth <= 0 or self.debt_usd <= 0:
            raise ValueError("Loan collateral and debt must be positive.")


class LoanNotFoundError(Exception):
    """Raised when a borrower has no open loan."""


class PositionNotLiquidatableError(Exception):
    """Raised when liquidation is attempted on a still-healthy position."""


class InsufficientRepaymentError(Exception):
    """Raised when a flash-loan action cannot return its borrowed principal."""


class LendingProtocol:
    """A toy lending protocol that liquidates undercollateralised loans.

    Its one deliberate flaw: by default it prices collateral from
    ``pool.spot_price`` -- a single, on-chain, manipulable number -- unless
    a ``price_source`` (e.g. a ``MedianOracle``) is supplied instead.
    """

    def __init__(
        self, pool: AMMPool, liquidation_ratio: float = 1.5, price_source=None
    ) -> None:
        """Configure the protocol's price source and liquidation threshold.

        Args:
            pool: The AMM this protocol reads a price from by default.
            liquidation_ratio: Minimum healthy collateral ratio. Must be
                positive.
            price_source: Optional object exposing a ``.price`` property.
                When given, it is trusted instead of ``pool.spot_price``.

        Raises:
            ValueError: If ``liquidation_ratio`` is not positive.
        """
        if liquidation_ratio <= 0:
            raise ValueError("Liquidation ratio must be positive.")
        self.pool = pool
        self.liquidation_ratio = liquidation_ratio
        self.price_source = price_source
        self.loans: dict[str, Loan] = {}

    def add_loan(self, borrower: str, loan: Loan) -> None:
        """Register an open loan for a borrower.

        Args:
            borrower: Non-empty borrower identifier.
            loan: The loan to register.

        Raises:
            ValueError: If ``borrower`` is empty.
        """
        if not borrower:
            raise ValueError("Borrower name must be non-empty.")
        self.loans[borrower] = loan

    def collateral_ratio(self, loan: Loan) -> float:
        """Return collateral value divided by debt, using the configured price source."""
        price = (
            self.price_source.price
            if self.price_source is not None
            else self.pool.spot_price
        )
        return loan.collateral_eth * price / loan.debt_usd

    def liquidate(self, borrower: str) -> Loan:
        """Seize and remove a borrower's loan if it is unhealthy.

        Args:
            borrower: The borrower to liquidate.

        Returns:
            The removed ``Loan``.

        Raises:
            LoanNotFoundError: If ``borrower`` has no open loan.
            PositionNotLiquidatableError: If the loan's collateral ratio
                is still at or above ``self.liquidation_ratio``.
        """
        if borrower not in self.loans:
            raise LoanNotFoundError(f"No loan for {borrower}.")
        loan = self.loans[borrower]
        if self.collateral_ratio(loan) >= self.liquidation_ratio:
            raise PositionNotLiquidatableError("Position is still healthy.")
        return self.loans.pop(borrower)

In [14]:
baseline_pool = AMMPool(50.0, 100_000.0)
baseline_protocol = LendingProtocol(baseline_pool)
victim_loan = Loan(collateral_eth=10.0, debt_usd=12_000.0)
baseline_protocol.add_loan("victim", victim_loan)
baseline_ratio = baseline_protocol.collateral_ratio(victim_loan)

print(f"At ${baseline_pool.spot_price:,.2f}/ETH, victim collateral ratio: {baseline_ratio:.2f}")
print("Liquidation threshold: 1.50; verdict: HEALTHY")


At $2,000.00/ETH, victim collateral ratio: 1.67
Liquidation threshold: 1.50; verdict: HEALTHY


**Read the result:** $20,000 of collateral / $12,000 debt is 1.67, so the position is healthy before anyone touches the pool. The coming liquidation is not the victim changing their loan. It is the protocol trusting a temporary photograph of the puddle.


## 5. Flash loans and the heist

A flash-loan provider lends ETH only if the principal is back by the end of the same transaction: enormous buying power with a same-block return policy. This toy provider charges no fee. It snapshots everything it touches and restores that snapshot if anything fails. Only ETH left after repayment counts as attacker profit.

The provider has 1,000 ETH; the attacker borrows 40. The pool starts at 50 ETH and $100,000.

We will submit that heist as a notebook 5 `Transaction`, gossip it through a `Network`, and let a proposer include it only if it is in their local mempool. Then we watch what the payload *does*.

> Pause and predict: after paying the $12,000 debt and reversing the dump, how much ETH remains once the 40 ETH principal is repaid?


In [17]:
@dataclass(frozen=True)
class WorldSnapshot:
    """A pre-action copy of every modeled object a flash loan might mutate.

    Attributes:
        pool_eth: AMM ETH reserve before the action.
        pool_usd: AMM USD reserve before the action.
        loans: Copy of the lending protocol's open loans before the action.
        provider_eth: Flash-loan provider liquidity before the action.
    """

    pool_eth: float
    pool_usd: float
    loans: dict[str, Loan]
    provider_eth: float


@dataclass(frozen=True)
class AttackTrace:
    """A structured record of what a flash-loan attack actually did.

    Attributes:
        borrowed_eth: Principal borrowed for the transaction.
        usd_from_dump: USD received from selling the borrowed ETH.
        manipulated_price: AMM spot price immediately after the dump.
        victim_ratio: Victim's collateral ratio at the manipulated price.
        debt_paid_usd: USD paid to liquidate the victim.
        collateral_seized: ETH collateral seized from the victim.
        eth_bought_back: ETH bought back with the leftover USD.
        eth_before_repayment: Total attacker ETH before repaying principal.
        principal_repaid: ETH principal returned to the flash-loan provider.
    """

    borrowed_eth: float
    usd_from_dump: float
    manipulated_price: float
    victim_ratio: float
    debt_paid_usd: float
    collateral_seized: float
    eth_bought_back: float
    eth_before_repayment: float
    principal_repaid: float


class FlashLoanProvider:
    """Lends ETH that must be repaid before the same call returns.

    Snapshots every modeled object the action touches beforehand and
    restores that snapshot if the action raises or fails to repay --
    modelling atomic all-or-nothing execution without a real EVM.
    """

    def __init__(self, eth_available: float) -> None:
        """Set the provider's lendable liquidity.

        Args:
            eth_available: ETH the provider can lend. Must be positive.

        Raises:
            ValueError: If ``eth_available`` is not positive.
        """
        if eth_available <= 0:
            raise ValueError("Flash-loan liquidity must be positive.")
        self.eth_available = eth_available

    def execute(self, amount_eth, action, pool, protocol):
        """Lend ``amount_eth``, run ``action``, then commit or roll back.

        Args:
            amount_eth: ETH to lend. Must be positive and available.
            action: Callable taking the borrowed ETH amount and returning
                ``(eth_before_repayment, trace)``.
            pool: The AMM the action may mutate (snapshotted first).
            protocol: The lending protocol the action may mutate
                (snapshotted first).

        Returns:
            ``(profit_eth, trace)`` where ``profit_eth`` is whatever ETH
            remains after repaying the principal.

        Raises:
            ValueError: If ``amount_eth`` is not positive or exceeds
                available liquidity.
            InsufficientRepaymentError: If ``action`` cannot return at
                least ``amount_eth``. In this case (or any other
                exception from ``action``), ``pool``, ``protocol``, and
                this provider's balance are restored from the snapshot.
        """
        if amount_eth <= 0 or amount_eth > self.eth_available:
            raise ValueError("Flash-loan amount is unavailable.")
        snapshot = WorldSnapshot(
            pool.eth_reserve,
            pool.usd_reserve,
            dict(protocol.loans),
            self.eth_available,
        )
        self.eth_available -= amount_eth
        try:
            eth_before_repayment, trace = action(amount_eth)
            if eth_before_repayment < amount_eth:
                raise InsufficientRepaymentError(
                    f"Only {eth_before_repayment:.4f} ETH available to repay "
                    f"{amount_eth:.4f} ETH."
                )
            self.eth_available += amount_eth
            return eth_before_repayment - amount_eth, trace
        except Exception:
            pool.eth_reserve = snapshot.pool_eth
            pool.usd_reserve = snapshot.pool_usd
            protocol.loans = dict(snapshot.loans)
            self.eth_available = snapshot.provider_eth
            raise


def run_flash_attack(amount_eth: float, pool: AMMPool, protocol: LendingProtocol, victim: str) -> tuple[float, AttackTrace]:
    """Borrow, dump, liquidate, buy back, and report the resulting trace.

    Args:
        amount_eth: ETH to borrow and dump into ``pool``.
        pool: The AMM to manipulate.
        protocol: The lending protocol whose price source will be read.
        victim: Borrower name to liquidate once the price is manipulated.

    Returns:
        ``(eth_before_repayment, trace)`` -- total attacker ETH before
        repaying the flash-loan principal, and a structured trace of the
        attack's steps.

    Raises:
        PositionNotLiquidatableError: If the victim is not actually
            liquidatable at the manipulated price (propagates from
            ``protocol.liquidate``).
        InsufficientRepaymentError: If the liquidation proceeds leave no
            USD for the buyback swap.
    """
    print(f"STEP 1 — BORROW: {amount_eth:.2f} ETH arrives temporarily.")
    usd_from_dump = pool.swap_eth_for_usd(amount_eth)
    manipulated_price = pool.spot_price
    print(
        f"STEP 2 — DUMP: sell {amount_eth:.2f} ETH for ${usd_from_dump:,.2f}; "
        f"AMM price becomes ${manipulated_price:,.2f}/ETH."
    )
    victim_loan = protocol.loans[victim]
    victim_ratio = protocol.collateral_ratio(victim_loan)
    seized_loan = protocol.liquidate(victim)
    debt_paid_usd = seized_loan.debt_usd
    collateral_seized = seized_loan.collateral_eth
    print(
        f"STEP 3 — LIQUIDATE: victim ratio is {victim_ratio:.2f}; "
        f"pay ${debt_paid_usd:,.2f} debt and seize {collateral_seized:.2f} ETH."
    )
    usd_for_buyback = usd_from_dump - debt_paid_usd
    if usd_for_buyback <= 0:
        raise InsufficientRepaymentError("Liquidation leaves no USD for the buyback.")
    eth_bought_back = pool.swap_usd_for_eth(usd_for_buyback)
    eth_before_repayment = eth_bought_back + collateral_seized
    print(
        f"STEP 4 — BUY BACK: remaining ${usd_for_buyback:,.2f} buys back "
        f"{eth_bought_back:.2f} ETH; attacker holds {eth_before_repayment:.2f} ETH."
    )
    print(f"STEP 5 — REPAY: return {amount_eth:.2f} ETH principal to the provider.")
    trace = AttackTrace(
        borrowed_eth=amount_eth,
        usd_from_dump=usd_from_dump,
        manipulated_price=manipulated_price,
        victim_ratio=victim_ratio,
        debt_paid_usd=debt_paid_usd,
        collateral_seized=collateral_seized,
        eth_bought_back=eth_bought_back,
        eth_before_repayment=eth_before_repayment,
        principal_repaid=amount_eth,
    )
    return eth_before_repayment, trace

### The heist is still a `Transaction`

Notebook 5 already has the waiting room. We import that same `Transaction` and `Network` from `blockchain_lib.mempool` — not a second, quieter inbox — and subclass `Network` so a proposer can include a transaction only if it is in their local mempool.

The flash-loan payload has not executed yet. First it has to be gossiped, then included. A node that missed the gossip cannot propose it, no matter how profitable it looks.

> Pause and predict: if SketchyGuy-Node originates the attack, can a peer who has not received it include it?


In [19]:
import random
import sys
from pathlib import Path


def _repo_root(marker: str = "pyproject.toml") -> Path:
    """Walk upward from the current working directory to find the repo root."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not find {marker} above {Path.cwd()}")


_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Block, Blockchain, Validator


class ProposingNetwork(Network):
    """Notebook 5's gossip network, plus inclusion into a PoS block.

    A proposer may include only a transaction sitting in *their* mempool.
    After inclusion the transaction is dropped from every local inbox:
    it is no longer waiting; it is history (or a receipt of an attempt).
    """

    def include(
        self, proposer: str, tx_id: str, chain: Blockchain
    ) -> tuple[Transaction, Block]:
        """Include ``tx_id`` from ``proposer``'s mempool and append a block.

        Args:
            proposer: Node that believes it can propose this transaction.
            tx_id: Identifier of the transaction to include.
            chain: The PoS chain that will record the inclusion.

        Returns:
            The included ``Transaction`` and the new ``Block``.

        Raises:
            ValueError: If ``proposer`` is unknown, or ``tx_id`` is not
                in that proposer's local mempool, or the candidate does
                not build on the current tip.
        """
        transaction = self.get(proposer, tx_id)
        if transaction is None:
            raise ValueError(
                f"{proposer} cannot include {tx_id}: "
                "it is not in their local mempool."
            )
        block = chain.propose_candidate(transaction.description, proposer)
        chain.accept_candidate(block)
        self.drop(tx_id)
        return transaction, block


@dataclass(frozen=True)
class TransactionReceipt:
    """Per-transaction execution result for a block already on the chain.

    Inclusion says the transaction was recorded and executed. ``status``
    says separately whether its state changes survived — the same
    distinction notebook 5 drew between inclusion and confirmation, and
    that real chains keep between "mined" and "succeeded."

    Attributes:
        transaction: The notebook 5 ``Transaction`` that was included.
        block: The chain block that included it.
        status: ``"SUCCESS"`` or ``"REVERTED"``.
        gas_used: Gas consumed by execution, regardless of status.
        state_effect: What (if anything) changed in modeled state.
    """

    transaction: Transaction
    block: Block
    status: str
    gas_used: int
    state_effect: str


In [20]:
nodes = ["Node A", "SketchyGuy-Node", "Emma-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("SketchyGuy-Node", 80),
    Validator("Emma-Node", 70),
    Validator("Farid-Node", 50),
]
network = ProposingNetwork(nodes, random.Random(7))
chain = Blockchain(validators)
receipts: list[TransactionReceipt] = []

attack_tx = Transaction("tx-flash-attack", "Flash-loan oracle manipulation")
network.broadcast(attack_tx, origin="SketchyGuy-Node")

print("Each node has its own mempool after the attack is gossiped:")
for node in nodes:
    print(f"  {node}: {network.mempool_ids(node)}")

missing = [name for name in nodes if network.get(name, attack_tx.tx_id) is None]
if missing:
    try:
        network.include(missing[0], attack_tx.tx_id, chain)
    except ValueError as error:
        print(f"\n{error}")

included_attack, attack_block = network.include(
    "SketchyGuy-Node", attack_tx.tx_id, chain
)
print(
    f"\n{attack_block.proposer} included {included_attack.tx_id} "
    f"in Block #{attack_block.index}."
)
print(
    "Mempools after inclusion: "
    + ", ".join(f"{name}={network.mempool_ids(name)}" for name in nodes)
)


Each node has its own mempool after the attack is gossiped:
  Node A: ['tx-flash-attack']
  SketchyGuy-Node: ['tx-flash-attack']
  Emma-Node: ['tx-flash-attack']
  Farid-Node: []

Farid-Node cannot include tx-flash-attack: it is not in their local mempool.

SketchyGuy-Node included tx-flash-attack in Block #1.
Mempools after inclusion: Node A=[], SketchyGuy-Node=[], Emma-Node=[], Farid-Node=[]


In [21]:
attack_pool = AMMPool(50.0, 100_000.0)
attack_protocol = LendingProtocol(attack_pool)
attack_protocol.add_loan("victim", Loan(10.0, 12_000.0))
provider = FlashLoanProvider(1_000.0)

profit_eth, attack_trace = provider.execute(
    40.0,
    lambda borrowed_eth: run_flash_attack(
        borrowed_eth, attack_pool, attack_protocol, "victim"
    ),
    attack_pool,
    attack_protocol,
)
assert attack_trace.eth_before_repayment == (
    attack_trace.principal_repaid + profit_eth
)
assert provider.eth_available == 1_000.0

print("Attack transaction: COMMITTED")
print(f"Provider liquidity after repayment: {provider.eth_available:,.2f} ETH")
print(f"Attacker profit: {profit_eth:.2f} ETH")

receipts.append(
    TransactionReceipt(
        included_attack,
        attack_block,
        "SUCCESS",
        310_000,
        "Attack state committed",
    )
)


STEP 1 — BORROW: 40.00 ETH arrives temporarily.
STEP 2 — DUMP: sell 40.00 ETH for $44,444.44; AMM price becomes $617.28/ETH.
STEP 3 — LIQUIDATE: victim ratio is 0.51; pay $12,000.00 debt and seize 10.00 ETH.
STEP 4 — BUY BACK: remaining $32,444.44 buys back 33.18 ETH; attacker holds 43.18 ETH.
STEP 5 — REPAY: return 40.00 ETH principal to the provider.
Attack transaction: COMMITTED
Provider liquidity after repayment: 1,000.00 ETH
Attacker profit: 3.18 ETH


**Read the result:** the dump produces $44,444.44 and makes the victim look unsafe at a 0.51 ratio. Paying the $12,000 debt leaves $32,444.44 for the reverse swap, which buys back about 33.18 ETH. Add the seized 10 ETH collateral, repay 40, and about **3.18 ETH** remains as profit. The provider is back at exactly 1,000 ETH.

Every step followed the rules. The rules asked the puddle for the price of the ocean. SketchyGuy-Node could include the payload only because it was already in that node's mempool; a peer who missed gossip could not have proposed it.


## 6. When one step fails, the whole transaction rewinds

Change only the victim: Victim2 has 40 ETH against the same $12,000 debt. Even after the 40 ETH dump, that position should stay healthy. The provider will still lend, the AMM will still be touched, and then liquidation will raise.

This is atomicity doing its actual job — not detecting villains, just refusing to leave a half-finished world around. The failed attempt is still a `Transaction`: it still has to be gossiped and included. Inclusion records the attempt. Execution status decides whether any state change survives.

> Pause and predict: after liquidation fails, which values should look exactly as they did before the flash loan?


In [24]:
failed_tx = Transaction("tx-flash-victim2", "Flash-loan against Victim2")
network.broadcast(failed_tx, origin="SketchyGuy-Node")
included_failed, failed_block = network.include(
    "SketchyGuy-Node", failed_tx.tx_id, chain
)
print(
    f"{failed_block.proposer} included {included_failed.tx_id} "
    f"in Block #{failed_block.index} (execution comes next)."
)

failed_pool = AMMPool(50.0, 100_000.0)
failed_protocol = LendingProtocol(failed_pool)
failed_protocol.add_loan("Victim2", Loan(40.0, 12_000.0))
failed_provider = FlashLoanProvider(1_000.0)
failed_before = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
failed_observation: dict[str, float] = {}

def failed_attack_action(amount_eth: float):
    failed_pool.swap_eth_for_usd(amount_eth)
    failed_observation["price"] = failed_pool.spot_price
    failed_observation["ratio"] = failed_protocol.collateral_ratio(
        failed_protocol.loans["Victim2"]
    )
    failed_protocol.liquidate("Victim2")
    raise RuntimeError("Liquidation should have raised first.")

try:
    failed_provider.execute(
        40.0, failed_attack_action, failed_pool, failed_protocol
    )
except PositionNotLiquidatableError:
    print(
        f"Attempted dump moved AMM price to "
        f"${failed_observation['price']:,.2f}/ETH."
    )
    print(
        f"Victim2 ratio after dump: {failed_observation['ratio']:.2f}; "
        "liquidation is rejected."
    )
    print("Attack transaction: REVERTED")

failed_after = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
rollback_verified = failed_before == failed_after
print(f"Rollback complete: {rollback_verified}")

receipts.append(
    TransactionReceipt(
        included_failed,
        failed_block,
        "REVERTED",
        185_000,
        "No state change",
    )
)


SketchyGuy-Node included tx-flash-victim2 in Block #2 (execution comes next).
Attempted dump moved AMM price to $617.28/ETH.
Victim2 ratio after dump: 2.06; liquidation is rejected.
Attack transaction: REVERTED
Rollback complete: True


**Read the result:** the AMM price briefly reached $617.28, but Victim2 still had a 2.06 ratio, so liquidation was rejected. The provider restored the pool, the loan list, and its own balance from the snapshot.

Atomicity stops half-finished state. It cannot stop a fully completed transaction from exploiting a bad price rule — that is what the first attack was. The Victim2 attempt was still gossiped and included; its receipt will say `REVERTED`. Consensus records the attempt. It does not make the price source honest.


## 7. Defense: stop asking the puddle

A median of independent reports is not the same observation as a thin AMM's reserves. Dump into a fresh pool exactly as before, but point the lending protocol at the `MedianOracle` from section 1.

The dump still happens. The pool still looks seasick. The protocol just refuses to take medical advice from the pool. Aggregation helps here because the protocol *stopped reading the AMM*, not because the AMM became honest.

The reports are $2,005, $1,995, and $2,000.

> Pause and predict: after the dump pushes the AMM to about $617/ETH, will the protocol liquidate the original 10 ETH / $12,000 loan if its oracle is still near $2,000?


In [27]:
defense_oracle = MedianOracle(
    [
        PriceReport("exchange-A", 2_005.0),
        PriceReport("exchange-B", 1_995.0),
        PriceReport("reference-feed", 2_000.0),
    ]
)
defense_pool = AMMPool(50.0, 100_000.0)
defense_protocol = LendingProtocol(
    defense_pool, price_source=defense_oracle
)
defense_victim_loan = Loan(10.0, 12_000.0)
defense_protocol.add_loan("defense-victim", defense_victim_loan)
defense_pool.swap_eth_for_usd(40.0)
defense_ratio = defense_protocol.collateral_ratio(defense_victim_loan)

try:
    defense_protocol.liquidate("defense-victim")
except PositionNotLiquidatableError:
    defense_rejected = True
else:
    defense_rejected = False

def defense_verdict(defense_rejected: bool) -> str:
    return "liquidation rejected" if defense_rejected else "liquidation executed"


print(f"Manipulated AMM spot price: ${defense_pool.spot_price:,.2f}/ETH")
print(
    f"Median oracle price: ${defense_oracle.price:,.2f}/ETH; "
    f"protocol ratio: {defense_ratio:.2f}"
)
print(f"Defense result: {defense_verdict(defense_rejected)}")


Manipulated AMM spot price: $617.28/ETH
Median oracle price: $2,000.00/ETH; protocol ratio: 1.67
Defense result: liquidation rejected


**Read the result:** the AMM really moved to $617.28, but the protocol read the median oracle's $2,000 and rejected the healthy loan. Same median idea as section 1, now plugged into the protocol that used to consult the puddle.

Median feeds, time-weighted averages, deeper liquidity, freshness limits, circuit breakers, and conservative parameters are layers of defense, not silver bullets.


## 8. Included is not the same as succeeded

Notebook 5 separated origin, gossip receipt, inclusion, and confirmation. Add one more column: **execution status**. Both attack transactions above were included in blocks. Only one of them left a state change. A reverted transaction can still consume gas and still occupy a block; consensus records the attempt. It does not make the price source honest.

> Pause and predict: which receipt was included but left no modeled state change?


In [30]:
print("Block | Transaction                      | Status   | Gas used | State effect")
for receipt in receipts:
    print(
        f"{receipt.block.index:>5} | {receipt.transaction.description:<32} "
        f"| {receipt.status:<8} | {receipt.gas_used:>8,} | {receipt.state_effect}"
    )
print("Included does not mean succeeded")
valid, message = chain.is_valid()
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(receipts)} inclusions)")


Block | Transaction                      | Status   | Gas used | State effect
    1 | Flash-loan oracle manipulation   | SUCCESS  |  310,000 | Attack state committed
    2 | Flash-loan against Victim2       | REVERTED |  185,000 | No state change
Included does not mean succeeded
Chain valid? True -- Chain is valid.
Canonical length: 3 (genesis + 2 inclusions)


**Read the result:** both payloads were gossiped through notebook 5's `Network`, included from a local mempool by `ProposingNetwork`, and recorded on the same `Blockchain` notebook 5 used. The successful heist committed. The Victim2 attempt reverted, so the snapshot survived, but its receipt is still in the chain and it still paid gas. Fork choice and validity do not audit the oracle.


## Takeaways

- **Mechanism:** an oracle is the input a contract treats as a fact about the world. **Not a guarantee:** that fact is true.
- **Mechanism:** a median of independent, fresh reports raises the cost of one liar. **Not a guarantee:** aggregation proves reality, or saves you if you still read a thin AMM.
- **Mechanism:** an AMM spot price honestly reflects its own reserves. **Not a guarantee:** that local ratio is "the market."
- **Mechanism:** a flash loan makes huge capital available until the transaction ends. **Not a guarantee:** the lending rule was a good idea.
- **Mechanism:** a flash-loan attack is a `Transaction` that must be gossiped and included from a local mempool. **Not a guarantee:** a proposer who missed gossip can include it, or inclusion means the heist succeeded.
- **Mechanism:** atomicity prevents half-finished state. **Not a guarantee:** a completed exploit of a bad price source gets rolled back out of fairness. Included is not the same as succeeded.
